In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_T_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "53234477 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.9,27.6,T0DayMin1,2,0,1
42667,96.7,28.3,T0DayMin1,2,0,1


# Compute mean/max T per day preceding measurement

In [3]:
df_deg_bins_day <- df_deg_bins %>%
                        group_by(Measured_date_CDC,cell_x,cell_y, Day_Min)%>%
                        summarize(T_min = min(Temp),
                                  T_mean = sum(N_hours*Temp)/24,
                                  T_max = max(Temp))
sprintf("%i x %i dataframe", nrow(df_deg_bins_day), ncol(df_deg_bins_day))
head(df_deg_bins_day,2)

`summarise()` has grouped output by 'Measured_date_CDC', 'cell_x', 'cell_y'. You can override using the `.groups` argument.


[1] "5209710 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_min,T_mean,T_max
<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>
42022,74,15.4,1,18,24.45833,31
42022,74,15.4,2,18,24.04167,31


In [4]:
data.frame(df_deg_bins_day) %>% 
        select(T_min,T_mean,T_max)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic     N     Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
------------------------------------------------------------------------
T_min     5,209,710 18.02   7.27    -27      13      20      24     33  
T_mean    5,209,710 22.96   6.73   -19.67  18.79   24.17   27.67   39.29
T_max     5,209,710 28.22   6.84    -15      24      29      33     45  
------------------------------------------------------------------------


# Categorize T per hour into bins

In [5]:
thresholds_T_min <- c(-27,11,13,15,17,19,21,23,25,33)
thresholds_T_mean <- c(-20,17,21,23,25,27,29,31,33,45)
thresholds_T_max <- c(-15,21,23,25,27,29,31,33,35,45)

In [6]:
df_deg_bins_day_bins <- df_deg_bins_day %>%
                            mutate(T_min_bin = cut(T_min, breaks = thresholds_T_min, 
                                                 labels = c("T_MinBin_inf11","T_MinBin_11_13",
                                                            "T_MinBin_13_15","T_MinBin_15_17","T_MinBin_17_19",
                                                            "T_MinBin_19_21","T_MinBin_21_23","T_MinBin_23_25",
                                                            "T_MinBin_sup25"),
                                                 include.lowest = TRUE),
                                   T_mean_bin = cut(T_mean, breaks = thresholds_T_mean, 
                                                 labels = c("T_MeanBin_inf17","T_MeanBin_17_21",
                                                            "T_MeanBin_21_23","T_MeanBin_23_25","T_MeanBin_25_27",
                                                            "T_MeanBin_27_29","T_MeanBin_29_31","T_MeanBin_31_33",
                                                            "T_MeanBin_sup33"),
                                                 include.lowest = TRUE),
                                   T_max_bin = cut(T_max, breaks = thresholds_T_max, 
                                                 labels = c("T_MaxBin_inf21","T_MaxBin_21_23","T_MaxBin_23_25",
                                                            "T_MaxBin_25_27","T_MaxBin_27_29","T_MaxBin_29_31",
                                                            "T_MaxBin_31_33","T_MaxBin_33_35",
                                                            "T_MaxBin_sup35"),
                                                 include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(df_deg_bins_day_bins), ncol(df_deg_bins_day_bins))
head(df_deg_bins_day_bins,2)

[1] "5209710 x 10 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_min,T_mean,T_max,T_min_bin,T_mean_bin,T_max_bin
<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<fct>,<fct>,<fct>
42022,74,15.4,1,18,24.45833,31,T_MinBin_17_19,T_MeanBin_23_25,T_MaxBin_29_31
42022,74,15.4,2,18,24.04167,31,T_MinBin_17_19,T_MeanBin_23_25,T_MaxBin_29_31


In [7]:
rm(df_deg_bins,df_deg_bins_day)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1603271,85.7,38090384,2034.3,47612980,2542.9
Vcells,39766789,303.4,617921631,4714.4,771774333,5888.2


# Pivot wider

## Min temperature

In [8]:
df_deg_min_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,T_min_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = T_min_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_min_wide), ncol(df_deg_min_wide))
head(df_deg_min_wide,2)

[1] "5209710 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_MinBin_17_19,T_MinBin_15_17,T_MinBin_19_21,T_MinBin_21_23,T_MinBin_inf11,T_MinBin_11_13,T_MinBin_13_15,T_MinBin_23_25,T_MinBin_sup25
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.4,1,1,0,0,0,0,0,0,0,0
42022,74,15.4,2,1,0,0,0,0,0,0,0,0


In [9]:
df_deg_min_Ndays_30 <- df_deg_min_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_30), ncol(df_deg_min_Ndays_30))
head(df_deg_min_Ndays_30,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MinBin_17_19_Ndays_on30,T_MinBin_15_17_Ndays_on30,T_MinBin_19_21_Ndays_on30,T_MinBin_21_23_Ndays_on30,T_MinBin_inf11_Ndays_on30,T_MinBin_11_13_Ndays_on30,T_MinBin_13_15_Ndays_on30,T_MinBin_23_25_Ndays_on30,T_MinBin_sup25_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,17,7,5,1,0,0,0,0,0
42022,74.1,15.2,15,5,9,1,0,0,0,0,0


In [10]:
df_deg_min_Ndays_20 <- df_deg_min_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_20), ncol(df_deg_min_Ndays_20))
head(df_deg_min_Ndays_20,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MinBin_17_19_Ndays_on20,T_MinBin_15_17_Ndays_on20,T_MinBin_19_21_Ndays_on20,T_MinBin_21_23_Ndays_on20,T_MinBin_inf11_Ndays_on20,T_MinBin_11_13_Ndays_on20,T_MinBin_13_15_Ndays_on20,T_MinBin_23_25_Ndays_on20,T_MinBin_sup25_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,9,7,4,0,0,0,0,0,0
42022,74.1,15.2,10,5,5,0,0,0,0,0,0


In [11]:
df_deg_min_Ndays_10 <- df_deg_min_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_10), ncol(df_deg_min_Ndays_10))
head(df_deg_min_Ndays_10,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MinBin_17_19_Ndays_on10,T_MinBin_15_17_Ndays_on10,T_MinBin_19_21_Ndays_on10,T_MinBin_21_23_Ndays_on10,T_MinBin_inf11_Ndays_on10,T_MinBin_11_13_Ndays_on10,T_MinBin_13_15_Ndays_on10,T_MinBin_23_25_Ndays_on10,T_MinBin_sup25_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,3,7,0,0,0,0,0,0,0
42022,74.1,15.2,5,5,0,0,0,0,0,0,0


In [12]:
rm(df_deg_min_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1618917,86.5,30472308,1627.4,47612980,2542.9
Vcells,45788537,349.4,494337305,3771.5,771774333,5888.2


## Mean temperature

In [13]:
df_deg_mean_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,T_mean_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = T_mean_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_mean_wide), ncol(df_deg_mean_wide))
head(df_deg_mean_wide,2)

[1] "5209710 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_MeanBin_23_25,T_MeanBin_21_23,T_MeanBin_25_27,T_MeanBin_inf17,T_MeanBin_17_21,T_MeanBin_27_29,T_MeanBin_29_31,T_MeanBin_31_33,T_MeanBin_sup33
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.4,1,1,0,0,0,0,0,0,0,0
42022,74,15.4,2,1,0,0,0,0,0,0,0,0


In [14]:
df_deg_mean_Ndays_30 <- df_deg_mean_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_30), ncol(df_deg_mean_Ndays_30))
head(df_deg_mean_Ndays_30,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MeanBin_23_25_Ndays_on30,T_MeanBin_21_23_Ndays_on30,T_MeanBin_25_27_Ndays_on30,T_MeanBin_inf17_Ndays_on30,T_MeanBin_17_21_Ndays_on30,T_MeanBin_27_29_Ndays_on30,T_MeanBin_29_31_Ndays_on30,T_MeanBin_31_33_Ndays_on30,T_MeanBin_sup33_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,23,6,1,0,0,0,0,0,0
42022,74.1,15.2,24,3,3,0,0,0,0,0,0


In [15]:
df_deg_mean_Ndays_20 <- df_deg_mean_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_20), ncol(df_deg_mean_Ndays_20))
head(df_deg_mean_Ndays_20,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MeanBin_23_25_Ndays_on20,T_MeanBin_21_23_Ndays_on20,T_MeanBin_25_27_Ndays_on20,T_MeanBin_inf17_Ndays_on20,T_MeanBin_17_21_Ndays_on20,T_MeanBin_27_29_Ndays_on20,T_MeanBin_29_31_Ndays_on20,T_MeanBin_31_33_Ndays_on20,T_MeanBin_sup33_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,14,6,0,0,0,0,0,0,0
42022,74.1,15.2,16,3,1,0,0,0,0,0,0


In [16]:
df_deg_mean_Ndays_10 <- df_deg_mean_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_10), ncol(df_deg_mean_Ndays_10))
head(df_deg_mean_Ndays_10,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MeanBin_23_25_Ndays_on10,T_MeanBin_21_23_Ndays_on10,T_MeanBin_25_27_Ndays_on10,T_MeanBin_inf17_Ndays_on10,T_MeanBin_17_21_Ndays_on10,T_MeanBin_27_29_Ndays_on10,T_MeanBin_29_31_Ndays_on10,T_MeanBin_31_33_Ndays_on10,T_MeanBin_sup33_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,7,3,0,0,0,0,0,0,0
42022,74.1,15.2,10,0,0,0,0,0,0,0,0


In [17]:
rm(df_deg_mean_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1619089,86.5,19502278,1041.6,47612980,2542.9
Vcells,51779935,395.1,395469844,3017.2,771774333,5888.2


## Max temperature

In [18]:
df_deg_max_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,T_max_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = T_max_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_max_wide), ncol(df_deg_max_wide))
head(df_deg_max_wide,2)

[1] "5209710 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,T_MaxBin_29_31,T_MaxBin_31_33,T_MaxBin_27_29,T_MaxBin_25_27,T_MaxBin_23_25,T_MaxBin_inf21,T_MaxBin_21_23,T_MaxBin_33_35,T_MaxBin_sup35
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.4,1,1,0,0,0,0,0,0,0,0
42022,74,15.4,2,1,0,0,0,0,0,0,0,0


In [19]:
df_deg_max_Ndays_30 <- df_deg_max_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_30), ncol(df_deg_max_Ndays_30))
head(df_deg_max_Ndays_30,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MaxBin_29_31_Ndays_on30,T_MaxBin_31_33_Ndays_on30,T_MaxBin_27_29_Ndays_on30,T_MaxBin_25_27_Ndays_on30,T_MaxBin_23_25_Ndays_on30,T_MaxBin_inf21_Ndays_on30,T_MaxBin_21_23_Ndays_on30,T_MaxBin_33_35_Ndays_on30,T_MaxBin_sup35_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,21,3,2,3,1,0,0,0,0
42022,74.1,15.2,19,5,2,4,0,0,0,0,0


In [20]:
df_deg_max_Ndays_20 <- df_deg_max_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_20), ncol(df_deg_max_Ndays_20))
head(df_deg_max_Ndays_20,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MaxBin_29_31_Ndays_on20,T_MaxBin_31_33_Ndays_on20,T_MaxBin_27_29_Ndays_on20,T_MaxBin_25_27_Ndays_on20,T_MaxBin_23_25_Ndays_on20,T_MaxBin_inf21_Ndays_on20,T_MaxBin_21_23_Ndays_on20,T_MaxBin_33_35_Ndays_on20,T_MaxBin_sup35_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,11,3,2,3,1,0,0,0,0
42022,74.1,15.2,9,5,2,4,0,0,0,0,0


In [21]:
df_deg_max_Ndays_10 <- df_deg_max_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_10), ncol(df_deg_max_Ndays_10))
head(df_deg_max_Ndays_10,2)

[1] "173657 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,T_MaxBin_29_31_Ndays_on10,T_MaxBin_31_33_Ndays_on10,T_MaxBin_27_29_Ndays_on10,T_MaxBin_25_27_Ndays_on10,T_MaxBin_23_25_Ndays_on10,T_MaxBin_inf21_Ndays_on10,T_MaxBin_21_23_Ndays_on10,T_MaxBin_33_35_Ndays_on10,T_MaxBin_sup35_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74.0,15.4,7,3,0,0,0,0,0,0,0
42022,74.1,15.2,5,5,0,0,0,0,0,0,0


In [22]:
rm(df_deg_max_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1619267,86.5,15601823,833.3,47612980,2542.9
Vcells,57771341,440.8,316375876,2413.8,771774333,5888.2


# Merge

In [23]:
result <- data.frame(df_deg_mean_Ndays_30) %>%
                    left_join(df_deg_mean_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_mean_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_30, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_max_Ndays_30, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_max_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_max_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "173657 x 84 dataframe"

,Measured_date_CDC,cell_x,cell_y,T_MeanBin_23_25_Ndays_on30,T_MeanBin_21_23_Ndays_on30,T_MeanBin_25_27_Ndays_on30,T_MeanBin_inf17_Ndays_on30,T_MeanBin_17_21_Ndays_on30,T_MeanBin_27_29_Ndays_on30,T_MeanBin_29_31_Ndays_on30,T_MeanBin_31_33_Ndays_on30,T_MeanBin_sup33_Ndays_on30,T_MeanBin_23_25_Ndays_on20,T_MeanBin_21_23_Ndays_on20,T_MeanBin_25_27_Ndays_on20,T_MeanBin_inf17_Ndays_on20,T_MeanBin_17_21_Ndays_on20,T_MeanBin_27_29_Ndays_on20,T_MeanBin_29_31_Ndays_on20,T_MeanBin_31_33_Ndays_on20,T_MeanBin_sup33_Ndays_on20,T_MeanBin_23_25_Ndays_on10,T_MeanBin_21_23_Ndays_on10,T_MeanBin_25_27_Ndays_on10,T_MeanBin_inf17_Ndays_on10,T_MeanBin_17_21_Ndays_on10,T_MeanBin_27_29_Ndays_on10,T_MeanBin_29_31_Ndays_on10,T_MeanBin_31_33_Ndays_on10,T_MeanBin_sup33_Ndays_on10,T_MinBin_17_19_Ndays_on30,T_MinBin_15_17_Ndays_on30,T_MinBin_19_21_Ndays_on30,T_MinBin_21_23_Ndays_on30,T_MinBin_inf11_Ndays_on30,T_MinBin_11_13_Ndays_on30,T_MinBin_13_15_Ndays_on30,T_MinBin_23_25_Ndays_on30,T_MinBin_sup25_Ndays_on30,T_MinBin_17_19_Ndays_on20,T_MinBin_15_17_Ndays_on20,T_MinBin_19_21_Ndays_on20,T_MinBin_21_23_Ndays_on20,T_MinBin_inf11_Ndays_on20,T_MinBin_11_13_Ndays_on20,T_MinBin_13_15_Ndays_on20,T_MinBin_23_25_Ndays_on20,T_MinBin_sup25_Ndays_on20,T_MinBin_17_19_Ndays_on10,T_MinBin_15_17_Ndays_on10,T_MinBin_19_21_Ndays_on10,T_MinBin_21_23_Ndays_on10,T_MinBin_inf11_Ndays_on10,T_MinBin_11_13_Ndays_on10,T_MinBin_13_15_Ndays_on10,T_MinBin_23_25_Ndays_on10,T_MinBin_sup25_Ndays_on10,T_MaxBin_29_31_Ndays_on30,T_MaxBin_31_33_Ndays_on30,T_MaxBin_27_29_Ndays_on30,T_MaxBin_25_27_Ndays_on30,T_MaxBin_23_25_Ndays_on30,T_MaxBin_inf21_Ndays_on30,T_MaxBin_21_23_Ndays_on30,T_MaxBin_33_35_Ndays_on30,T_MaxBin_sup35_Ndays_on30,T_MaxBin_29_31_Ndays_on20,T_MaxBin_31_33_Ndays_on20,T_MaxBin_27_29_Ndays_on20,T_MaxBin_25_27_Ndays_on20,T_MaxBin_23_25_Ndays_on20,T_MaxBin_inf21_Ndays_on20,T_MaxBin_21_23_Ndays_on20,T_MaxBin_33_35_Ndays_on20,T_MaxBin_sup35_Ndays_on20,T_MaxBin_29_31_Ndays_on10,T_MaxBin_31_33_Ndays_on10,T_MaxBin_27_29_Ndays_on10,T_MaxBin_25_27_Ndays_on10,T_MaxBin_23_25_Ndays_on10,T_MaxBin_inf21_Ndays_on10,T_MaxBin_21_23_Ndays_on10,T_MaxBin_33_35_Ndays_on10,T_MaxBin_sup35_Ndays_on10
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,42022,74.0,15.4,23,6,1,0,0,0,0,0,0,14,6,0,0,0,0,0,0,0,7,3,0,0,0,0,0,0,0,17,7,5,1,0,0,0,0,0,9,7,4,0,0,0,0,0,0,3,7,0,0,0,0,0,0,0,21,3,2,3,1,0,0,0,0,11,3,2,3,1,0,0,0,0,7,3,0,0,0,0,0,0,0
2,42022,74.1,15.2,24,3,3,0,0,0,0,0,0,16,3,1,0,0,0,0,0,0,10,0,0,0,0,0,0,0,0,15,5,9,1,0,0,0,0,0,10,5,5,0,0,0,0,0,0,5,5,0,0,0,0,0,0,0,19,5,2,4,0,0,0,0,0,9,5,2,4,0,0,0,0,0,5,5,0,0,0,0,0,0,0


# Save

In [24]:
fwrite(result,"./output/df_T_cells_CDC_Ndays.csv")